# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.

## Setup

In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, json
from typing import Annotated, Literal
from pydantic import BaseModel, ConfigDict
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, PromptAgentDefinitionTextOptions, TextResponseFormatJsonSchema,FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput
from agents import Tool

In [ ]:

project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition is the core of the planning design pattern. Instead of asking a single agent to
handle a complex request end-to-end, we break the problem into smaller, well-defined **subtasks**.
Each subtask is assigned to a specialist agent (e.g., flights, hotels, activities) with clear
priorities and dependency ordering.

This approach provides several benefits:
- **Clarity**: each subtask has a single responsibility.
- **Parallelism**: independent subtasks can run concurrently.
- **Reliability**: failures are isolated to individual subtasks.
- **Budget tracking**: costs are estimated per subtask and rolled up.

In [ ]:
class TravelSubTask(BaseModel):
    model_config = ConfigDict(extra="forbid")

    task_id: int | None
    description: str | None
    assigned_agent: Literal["flight_agent", "hotel_agent", "activity_agent"] | None
    priority: Literal["high", "medium", "low"] | None
    dependencies: list[int]


class TravelPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    destination: str | None
    trip_duration_days: int | None
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int | None
    notes: str | None

schema = TravelPlan.model_json_schema()
print(schema["properties"].keys())
# print(schema["required"] or "")

## Creating a Planning Agent with Structured Output

The planning agent acts as a **front desk coordinator**. Given a high-level travel request it
produces a structured `TravelPlan` — decomposing the request into subtasks, setting priorities,
and identifying dependencies so that a concierge or execution layer can carry out the work.

In [ ]:
planning_agent = project_client.agents.create_version(
    agent_name="TravelPlanner",
    definition=PromptAgentDefinition(
        instructions="""You are a travel planning agent. When given a travel request:
            1. Break it into specific subtasks (flights, hotels, activities, logistics)
            2. Assign each subtask to the appropriate specialist agent
            3. Set priorities and identify dependencies between tasks
            4. Estimate the total budget""",
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        text=PromptAgentDefinitionTextOptions(
            format=TextResponseFormatJsonSchema(
                schema=TravelPlan.model_json_schema(),
                name="TravelPlan"
                )
        )
    ),
)

openai = project_client.get_openai_client()

response = openai.responses.create(
    input="I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    extra_body={
        "agent_reference": {
            "name": planning_agent.name,
            "type": "agent_reference",
        }
    },
)

print("Agent response:")
print(response.output_text)
if response.output_text:
    plan = TravelPlan(**json.loads(response.output_text))
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Executing a Plan with Specialist Tools

Once the front desk agent has produced a structured plan, the **concierge agent** executes it.
Each specialist tool handles one category of subtask (flights, hotels, activities). The concierge
iterates through the plan's subtasks in dependency order and dispatches each one to the
appropriate tool.

In [ ]:
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"

book_flight_tool = FunctionTool(
    name="book_flight",
    description="Search and book flights for the trip",
    parameters={
        "type": "object",
        "properties": {
        "destination": {"type": "string", "description": "The destination city"},
        "departure_date": {"type": "string", "description": "Departure date (YYYY-MM-DD)"},
        "return_date": {"type": "string", "description": "Return date (YYYY-MM-DD)"},
        },
        "required": ["destination", "departure_date", "return_date"],
        "additionalProperties": False
    },
    strict=True,
)

def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"

reserve_hotel_tool = FunctionTool(
    name="reserve_hotel",
    description="Reserve a hotel room in the destination city",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city for the hotel"},
            "check_in": {"type": "string", "description": "Check-in date (YYYY-MM-DD)"},
            "check_out": {"type": "string", "description": "Check-out date (YYYY-MM-DD)"},
            "guests": {"type": "integer", "description": "Number of guests"}
        },
        "required": ["city", "check_in", "check_out", "guests"],
        "additionalProperties": False
    },
    strict=True,
)

def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"

book_activity_tool = FunctionTool(
    name="book_activity",
    description="Book a tour, museum visit, or other activity",
    parameters={
        "type": "object",
        "properties": {
            "activity_name": {"type": "string", "description": "Name of the activity or tour"},
            "date": {"type": "string", "description": "Date of the activity (YYYY-MM-DD)"},
            "participants": {"type": "integer", "description": "Number of participants"}
        },
        "required": ["activity_name", "date", "participants"],
        "additionalProperties": False
    },
    strict=True,
)

tools: list[Tool] = [book_flight_tool, reserve_hotel_tool, book_activity_tool]

concierge_agent = project_client.agents.create_version(
    agent_name="Concierge",
    definition=PromptAgentDefinition(
        instructions=(
            "You are a travel concierge executing a structured travel plan. "
            "Use the available tools to fulfil each subtask. "
            "Work through the subtasks in order, respecting dependencies. "
            "Summarise the results when finished."
        ),
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        tools=tools
    ),
)

TOOL_HANDLERS = {
    "book_flight": book_flight,
    "reserve_hotel": reserve_hotel,
    "book_activity": book_activity
}

# Build a prompt from the plan produced above
if plan:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    print("\nExecuting plan with concierge agent...")   
    print(execution_prompt)

    exec_response = openai.responses.create(
        input=execution_prompt,
        extra_body={
            "agent_reference": {
                "name": concierge_agent.name,
                "type": "agent_reference",
            }
        }
    )

    tool_outputs = []
    for item in exec_response.output:
        if item.type != "function_call":
            continue
        tool_name = item.name
        args = json.loads(item.arguments or "{}")
        handler = TOOL_HANDLERS.get(tool_name)
        if handler:
            result = handler(**args)
            tool_outputs.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({ "result": result }),
                )
            )
            print(f"\nExecuted {tool_name} with args {args}, got output:\n{result}")

    final_response = openai.responses.create(
        input=tool_outputs,
        previous_response_id=exec_response.id,
        extra_body={
            "agent_reference": {
                "name": concierge_agent.name,
                "type": "agent_reference",
            }
        }
     )
    
    print("\nFinal agent summary:")
    print(final_response.output_text)

## Summary

In this lesson you learned the **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — A front desk planning agent breaks a complex travel request into
   structured subtasks using Pydantic models, assigning each to a specialist agent with priorities
   and dependencies.
2. **Structured Output** — By passing a `response_format` the agent returns a validated
   `TravelPlan` object instead of free-form text, making downstream processing reliable.
3. **Plan Execution** — A concierge agent iterates through the subtasks using specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to carry out the plan and report results.

This pattern separates *what to do* (planning) from *how to do it* (execution), making agents
more modular, testable, and easier to extend.